# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashir9099/flyrank-ml/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
import os
if not os.path.exists("flyrank-ml"):
    !git clone https://github.com/Hashir9099/flyrank-ml.git
os.chdir("flyrank-ml")
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

Cloning into 'flyrank-ml'...
remote: Enumerating objects: 206, done.
remote: Counting objects: 100% (206/206), done.
remote: Compressing objects: 100% (156/156), done.
remote: Total 206 (delta 89), reused 100 (delta 32), pack-reused 0 (from 0)
Receiving objects: 100% (206/206), 1.98 MiB | 10.70 MiB/s, done.
Resolving deltas: 100% (89/89), done.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: [paste from paper]
- Label source: [where does this claim's outcome variable come from?]
- Does the validation carry the claim: [does the split/sample size/method support it, or is
  it overclaiming?]

Finding 2: [paste from paper]
- Label source: [...]
- Does the validation carry the claim: [...]

Finding 1 — "The Anatomy of Growing Content" (Finding #1)
Claim: growing pages average 3.2K words vs 2.3K for declining pages; growing pages average
184 days old vs 230 for declining.

- Label source: `trend_direction`, defined from 30-day-vs-previous-30-day impression change
  (Up >10%, Down >10%, Stable within ±10%). This is the same label our own pipeline uses.
- Does the validation carry the claim: this is a large-sample (74.8K vs 45.6K pages) direct
  aggregate comparison, not a fitted model, so overfitting isn't the risk. But the paper itself
  flags it as "observational" — word count and age are correlated with each other and with
  many other page attributes (older pages tend to be shorter in this dataset per their own
  correlation matrix, r=-0.517 between age and word count), so it's not clear which of the two
  is doing the work, or whether both are proxies for something else like brand maturity. The
  claim is reasonable directionally but the "make it longer" recommendation isn't isolated
  from the "make it younger/refresh it" recommendation by this comparison alone.

Finding 2 — "What Predicts Growth?" (ML Appendix, logistic regression)
Claim: content age is the strongest negative predictor of growth; days-visible and recent
impressions are the strongest positive predictors. Reported at 71% holdout accuracy.

- Label source: same `trend_direction`-derived growth/decline split as Finding 1, but here
  it's the target of a fitted classifier rather than a raw comparison.
- Does the validation carry the claim: the paper reports an 80/20 holdout split, but doesn't
  say whether that split is random or grouped by brand — with 57 brands in the dataset,
  a random split risks the same optimism our own w06 experiment reproduced (0.947 random-split
  AUC dropping to 0.86 under a client-grouped split). 71% accuracy also isn't benchmarked
  against a naive baseline (e.g., always predicting "stable," since Stable is defined as a
  ±10% band and likely the largest class) — without that baseline, it's hard to tell how much
  71% actually beats "guess the majority class." The paper is appropriately cautious in its own
  language ("descriptive indicators... not direct instructions to optimize one variable in
  isolation"), which is the right framing, but the accuracy number alone doesn't tell a reader
  whether the split was honest.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Re-ran the model under two splits to check whether the reported performance survives an
honest evaluation:

- BEFORE (random split): AUC = 0.947
- AFTER (client-grouped split): AUC = 0.860

The drop (0.947 → 0.860) shows the random split was inflating performance — some of that
0.947 came from the model implicitly learning per-client patterns (seeing a client's other
pages in training and recognizing them in test), not from genuinely general signal. 0.860
under the grouped split is the more honest number: still a strong model, but the random-split
figure alone would have overstated how well this generalizes to a brand-new client.

In [6]:
from sklearn.model_selection import train_test_split, GroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

label_col = "trend_direction"
excluded = ["trend_direction", "trend_pct", "content_id", "client_id", "provider_used", "model_used"]

y = (df[label_col] == "down").astype(int)
X = df.drop(columns=excluded)
X_numeric = X.select_dtypes(include="number").fillna(X.select_dtypes(include="number").median())
X_cat = pd.get_dummies(X.select_dtypes(include="object"), dummy_na=True)
X_final = pd.concat([X_numeric, X_cat], axis=1)

# BEFORE: naive random split (what most people do first)
X_train, X_test, y_train, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)
model_naive = RandomForestClassifier(random_state=42).fit(X_train, y_train)
auc_naive = roc_auc_score(y_test, model_naive.predict_proba(X_test)[:, 1])
print("BEFORE (random split) AUC:", round(auc_naive, 3))

# AFTER: client-grouped split (no client's pages in both train and test)
gkf = GroupKFold(n_splits=5)
groups = df["client_id"]
train_idx, test_idx = next(gkf.split(X_final, y, groups=groups))
model_grouped = RandomForestClassifier(random_state=42).fit(X_final.iloc[train_idx], y.iloc[train_idx])
auc_grouped = roc_auc_score(y.iloc[test_idx], model_grouped.predict_proba(X_final.iloc[test_idx])[:, 1])
print("AFTER (client-grouped split) AUC:", round(auc_grouped, 3))

BEFORE (random split) AUC: 0.947
AFTER (client-grouped split) AUC: 0.86


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Correlation of each numeric feature with the label, on the final feature set (trend_direction,
trend_pct, and identifier columns already excluded):

- days_with_impressions: 0.190
- content_age_days: 0.164
- age_tier_order: 0.156
- impressions_last_30d: 0.094
- word_count: 0.084
- days_since_last_update: 0.081
- clicks_last_30d: 0.072
- char_count: 0.069
- sessions_last_30d: 0.064
- ctr: 0.062

None of these are suspiciously high (nothing near 1.0, unlike trend_pct which correlates
near-perfectly with the label by construction). The top feature, days_with_impressions
(0.19), is a legitimate signal — how many of the last 90 days had any impressions at all is
plausibly tied to declining visibility, not a leak of the label itself. No further exclusions
needed beyond what Section 1's data contract already ruled out.

In [7]:
# same hunt as w03_feature_leakage_check, applied to this final feature set
corrs = X_numeric.corrwith(y).abs().sort_values(ascending=False)
print(corrs.head(10))

days_with_impressions     0.190055
content_age_days          0.163882
age_tier_order            0.156142
impressions_last_30d      0.093980
word_count                0.084279
days_since_last_update    0.081383
clicks_last_30d           0.071935
char_count                0.068681
sessions_last_30d         0.063842
ctr                       0.061911
dtype: float64


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Bold version: "[your original overclaiming sentence]"

Rewritten: "In this sample, [X] was observed/measured to be associated with [Y] — directional,
decision-support signal, not a causal or algorithmic claim."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.